# 音のプログラミング 第3回: 周波数分析（FFTとスペクトログラム）

**前回までの復習:**
- 第1回：サイン波の生成、サンプリングの基礎
- 第2回：エンベロープ（ADSR）で音の時間変化をコントロール

**学習目標:**
- フーリエ変換（FFT）の基本を理解する
- 音の周波数成分を分析する
- 波形の違いと倍音構造の関係を体験する
- スペクトログラムで音の時間変化を可視化する
- フィルターの基礎（次回への準備）

**所要時間:** 90分

## 🛠️ 環境セットアップ

In [ ]:
# 🛠️ 環境セットアップ
import sys
try:
    import google.colab
    !pip install -q japanize-matplotlib
    !git clone -q https://github.com/ggszk/simple-sound-programming.git
    sys.path.append('/content/simple-sound-programming')
except ImportError:
    sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from audio_lib.notebook import setup_environment
setup_environment()

## 🛠️ 今回使用するライブラリ

In [ ]:
from audio_lib import sine_wave, square_wave, sawtooth_wave, adsr, AudioSignal
from audio_lib.notebook import play_sound, plot_waveform, plot_spectrum

## 🎯 実習1: サイン波のFFT — 最もシンプルな例

In [ ]:
# 440Hzのサイン波を作成してFFT分析
signal = sine_wave(440, 1.0)

# 波形とスペクトラムを表示
plot_waveform(signal, duration=0.01, title="440Hz サイン波")
plot_spectrum(signal, max_freq=2000, title="440Hz サイン波のスペクトラム")

440Hzにきれいな1本のピークが見えます。
サイン波は**単一の周波数成分**のみを持つ、最も純粋な波形です。

## 🎯 実習2: 和音のFFT — 複数の音を分離する

In [ ]:
# Cメジャーコード（ド・ミ・ソ）を作成
freq_c = 261.63  # C4
freq_e = 329.63  # E4
freq_g = 392.00  # G4

sig_c = sine_wave(freq_c, 1.0)
sig_e = sine_wave(freq_e, 1.0)
sig_g = sine_wave(freq_g, 1.0)

# 3つのサイン波を合成
chord = (sig_c + sig_e + sig_g) * (1.0 / 3)

display(play_sound(chord, "Cメジャーコード (C4 + E4 + G4)"))

# 波形を表示
plot_waveform(chord, duration=0.02, title="Cメジャーコード — 波形")

# FFTで分析（手動実装で学習）
data = chord.data
sr = chord.sample_rate
n = len(data)

# ハニング窓を適用してFFT
windowed = data * np.hanning(n)
fft_result = np.fft.fft(windowed)
fft_magnitude = np.abs(fft_result)
fft_freq = np.fft.fftfreq(n, 1/sr)

# 正の周波数のみプロット
pos_idx = fft_freq > 0
freq = fft_freq[pos_idx]
mag = fft_magnitude[pos_idx]

plt.figure(figsize=(12, 4))
plt.plot(freq[freq < 800], 20 * np.log10(mag[freq < 800] + 1e-10), 'g-', linewidth=2)
for f, name in [(freq_c, 'C4'), (freq_e, 'E4'), (freq_g, 'G4')]:
    plt.axvline(x=f, color='red', linestyle='--', alpha=0.7, label=f'{name} ({f}Hz)')
plt.title("Cメジャーコード — 周波数スペクトラム")
plt.xlabel("周波数 (Hz)")
plt.ylabel("振幅 (dB)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

複雑な波形でも、FFTを使えば構成する周波数成分を分離できます。
3つのピーク（261.63Hz, 329.63Hz, 392.00Hz）がはっきり見えますね。

## 🎯 実習3: 波形による倍音構造の違い

サイン波・矩形波・のこぎり波は、なぜ音色が違うのでしょうか？
その秘密は**倍音構造**にあります。

In [ ]:
# 3種類の波形を比較
freq = 220  # A3
duration = 1.0

waves = {
    "サイン波": sine_wave(freq, duration),
    "矩形波": square_wave(freq, duration),
    "のこぎり波": sawtooth_wave(freq, duration),
}

fig, axes = plt.subplots(3, 2, figsize=(14, 10))

for i, (name, sig) in enumerate(waves.items()):
    # 時間領域（波形）
    samples = int(0.01 * sig.sample_rate)
    time = np.linspace(0, 0.01, samples)
    axes[i, 0].plot(time, sig.data[:samples], 'b-', linewidth=2)
    axes[i, 0].set_title(f"{name} — 波形")
    axes[i, 0].set_xlabel("時間 (秒)")
    axes[i, 0].set_ylabel("振幅")
    axes[i, 0].grid(True, alpha=0.3)

    # 周波数領域（スペクトラム）
    n = len(sig)
    windowed = sig.data * np.hanning(n)
    fft_mag = np.abs(np.fft.fft(windowed))
    fft_freq = np.fft.fftfreq(n, 1/sig.sample_rate)
    pos = fft_freq > 0
    f = fft_freq[pos]
    m = fft_mag[pos]
    mask = f < 5000
    axes[i, 1].plot(f[mask], 20 * np.log10(m[mask] + 1e-10), 'g-', linewidth=1)
    axes[i, 1].set_title(f"{name} — スペクトラム")
    axes[i, 1].set_xlabel("周波数 (Hz)")
    axes[i, 1].set_ylabel("振幅 (dB)")
    axes[i, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 3種類の波形を聴き比べ
for name, sig in waves.items():
    display(play_sound(sig, name))

### 発見！

- **サイン波**: 基音（220Hz）のみ → 純粋で丸い音
- **矩形波**: 奇数倍音（220, 660, 1100, ...Hz）→ 電子的で中空的な音
- **のこぎり波**: 全倍音（220, 440, 660, ...Hz）→ 明るくシャープな音

この倍音の違いこそが**音色**（ティンバー）の正体です。

## 🎯 実習4: スペクトログラム — 時間×周波数の2Dマップ

FFTは「ある瞬間の周波数成分」を見るツールです。
**スペクトログラム**は、時間の流れに沿って周波数成分がどう変化するかを2Dマップとして表示します。

In [ ]:
# ドレミファソラシドの音階をADSRエンベロープ付きで作成
from audio_lib.synthesis.note_utils import note_to_frequency, note_name_to_number

scale = ["C4", "D4", "E4", "F4", "G4", "A4", "B4", "C5"]
scale_data = []

for note_name in scale:
    freq = note_to_frequency(note_name_to_number(note_name))
    sig = sine_wave(freq, 0.5)
    env = adsr(0.5, attack=0.02, decay=0.1, sustain=0.7, release=0.1)
    scale_data.append((sig * env).data)

scale_signal = AudioSignal(np.concatenate(scale_data), 44100)
display(play_sound(scale_signal, "ドレミファソラシド"))

# スペクトログラムを計算・表示
data = scale_signal.data
sr = scale_signal.sample_rate

frequencies, times, Sxx = sp_signal.spectrogram(data, fs=sr, nperseg=2048, noverlap=1024)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# 波形
time = np.linspace(0, len(data)/sr, len(data))
axes[0].plot(time, data, 'b-', linewidth=0.5)
axes[0].set_title("波形")
axes[0].set_xlabel("時間 (秒)")
axes[0].set_ylabel("振幅")
axes[0].grid(True, alpha=0.3)

# スペクトログラム
axes[1].pcolormesh(times, frequencies, 10 * np.log10(Sxx + 1e-10), shading='gouraud', cmap='magma')
axes[1].set_ylim(0, 1500)
axes[1].set_title("スペクトログラム")
axes[1].set_xlabel("時間 (秒)")
axes[1].set_ylabel("周波数 (Hz)")
plt.colorbar(axes[1].collections[0], ax=axes[1], label="パワー (dB)")

plt.tight_layout()
plt.show()

スペクトログラムで周波数が階段状に上がっていく様子が見えます。

## 🎯 実習5: 周波数分析の活用 — フィルターの予告

FFTを使えば、特定の周波数成分だけを取り除くことができます。
これが次回学ぶ「フィルター」の原理です。

In [ ]:
# のこぎり波をFFTし、1000Hz以上をカット（簡易ローパスフィルター）
original = sawtooth_wave(220, 1.0)

# FFT
n = len(original)
fft_data = np.fft.fft(original.data)
freqs = np.fft.fftfreq(n, 1/original.sample_rate)

# 1000Hz以上をゼロにする
cutoff = 1000
fft_filtered = fft_data.copy()
fft_filtered[np.abs(freqs) > cutoff] = 0

# 逆FFTで音声に戻す
filtered_data = np.real(np.fft.ifft(fft_filtered))
filtered_signal = AudioSignal(filtered_data, original.sample_rate)

display(play_sound(original, "元の音（のこぎり波）"))
display(play_sound(filtered_signal, f"フィルター後（{cutoff}Hz以上をカット）"))

# スペクトラム比較
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, data, title in [
    (axes[0], original.data, "元の音"),
    (axes[1], filtered_data, f"フィルター後（{cutoff}Hz以上カット）"),
]:
    n = len(data)
    windowed = data * np.hanning(n)
    mag = np.abs(np.fft.fft(windowed))
    freq = np.fft.fftfreq(n, 1/44100)
    pos = freq > 0
    f = freq[pos]
    m = mag[pos]
    mask = f < 5000
    ax.plot(f[mask], 20 * np.log10(m[mask] + 1e-10), 'g-', linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("周波数 (Hz)")
    ax.set_ylabel("振幅 (dB)")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

高い周波数成分をカットすると、音がまろやかになります。次回のフィルターの予習です！

## 🏆 チャレンジ課題

In [ ]:
# チャレンジ1: 自分だけの和音を作ってFFT分析してみよう
# 例: Amコード (A3 + C4 + E4)
freq_a = 220.00   # A3
freq_c = 261.63   # C4
freq_e = 329.63   # E4

chord = (sine_wave(freq_a, 1.0) + sine_wave(freq_c, 1.0) + sine_wave(freq_e, 1.0)) * (1.0 / 3)
display(play_sound(chord, "Amコード"))
plot_spectrum(chord, max_freq=1000, title="Amコードのスペクトラム")

In [ ]:
# チャレンジ2: カットオフ周波数を変えて音色の変化を体験しよう
original = sawtooth_wave(220, 1.0)

for cutoff in [500, 1000, 2000, 4000]:
    n = len(original)
    fft_data = np.fft.fft(original.data)
    freqs = np.fft.fftfreq(n, 1/original.sample_rate)
    fft_filtered = fft_data.copy()
    fft_filtered[np.abs(freqs) > cutoff] = 0
    filtered = AudioSignal(np.real(np.fft.ifft(fft_filtered)), original.sample_rate)
    display(play_sound(filtered, f"カットオフ {cutoff}Hz"))

## 📚 今日のまとめ

### 学んだこと
1. **FFT**（高速フーリエ変換）で音の周波数成分を分析できる
2. サイン波は単一周波数、矩形波・のこぎり波は倍音を含む
3. **倍音構造の違い**が音色（ティンバー）の違いを生む
4. **スペクトログラム**で時間×周波数の2Dマップを可視化できる
5. FFTの周波数成分を操作することでフィルターの原理を体験した

### 使ったテクニック
- `np.fft.fft()` / `np.fft.ifft()` — フーリエ変換 / 逆変換
- `np.fft.fftfreq()` — 周波数軸の計算
- `sp_signal.spectrogram()` — スペクトログラム計算
- `np.hanning()` — ハニング窓関数

### 次回予告
第4回では**フィルター**を本格的に学びます。
ローパスフィルターで音色をデザインし、シンプルな楽器音を作成します。